# Orders Lambda Batch

In [ ]:
import kafka

import pprint

from IPython.display import clear_output

import time

## Subscribe and read all the messages from the topic;  we can run this muliple times and always get all the messages; there can be an unlimited number of subscribers

In [ ]:
topic = "orders_pub_sub"

subscriber_read_all = kafka.KafkaConsumer(topic, 
                                          bootstrap_servers=['kafka:29092'], 
                                          auto_offset_reset='earliest')


In [ ]:
if subscriber_read_all.assignment():

    subscriber_read_all.seek_to_beginning()

message_list = []

while (True):
    
    poll_result = subscriber_read_all.poll(timeout_ms=500)
    
    if poll_result == {}:
        break

    items = poll_result.items()

    for (topic, messages) in items:
    
        for message in messages:
            
            message_list.append([message.offset, message.value])
            
i = 0

for message in message_list:
    
    if (i < 5) or (i > len(message_list) - 6):
        print("Offset:", message[0], "   Value:", message[1][:75])
   
    if i == 5:
        print("\n... only showing a max of first 5 and max of last 5 ... \n")
        
    i += 1
    
    

## Subscribe and read in batch mode;  read all the messages from the topic the first time we read; read only new messages on subsequent reads; we have a defined batch interval, such as every day, every hour, every 10 minutes, every 1 minute; here we will use 5 seconds not to waste time waiting;  Zookeeper will keep track of the offsets for us

In [ ]:
topic = "orders_pub_sub"

subscriber_batch = kafka.KafkaConsumer(topic, 
                                       bootstrap_servers=['kafka:29092'], 
                                       auto_offset_reset='earliest')


In [ ]:
batch_time_interval = 5.0

if subscriber_batch.assignment():

    subscriber_batch.seek_to_beginning()

batch_number = 1

message_list = []

while (True):
    
    poll_result = subscriber_batch.poll(timeout_ms=500)
    
    if poll_result == {}:
        
        if len(message_list) > 0:
            
            clear_output(wait=True)
            
            print("\n=================================")
            print("   Orders Lambda Batch Process")
            print("=================================\n")
            print("\n------------------------")
            print("   Batch ", batch_number)
            print("------------------------\n\n")
            
            for message in message_list:
                
                print("Offset:", message[0], "   Value:", message[1][:75])
                
            message_list = []
            
            batch_number += 1
            
        time.sleep(batch_time_interval)
              
    else:

        items = poll_result.items()

        for (topic, messages) in items:
    
            for message in messages:
        
                message_list.append([message.offset, message.value])


## You try it - demonstrate that 2 or more subscribers can subscribe to the same topic at the same time and both receive the same data;  make 1 or more copies of orders_lambda_batch and run multiple subscribers, both reading all and reading in batch mode